<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Lab 07 · Constructing the VIX from SPX Option Data

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the lab examples in a Colab-ready format so that you can
run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the lab text for detailed explanations and context.


## Project Setup
Set the project root so local data files, helper modules, and figure scripts
resolve correctly from `notebooks/labs/`.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks/labs"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

## Loading the Option Snapshot
The dataset contains SPX call and put quotes across maturities and strikes.


In [ ]:
from code.labs.lab07_vix_construction import load_option_snapshot

In [ ]:
pricing_date, options = load_option_snapshot()

In [ ]:
pricing_date.date()

In [ ]:
options.shape

## Choosing the VIX Maturities
The VIX targets thirty calendar days.


In [ ]:
from code.labs.lab07_vix_construction import available_expiries

In [ ]:
available_expiries(options).head(6)

In [ ]:
from code.labs.lab07_vix_construction import select_vix_expiries

In [ ]:
selected = select_vix_expiries(options)

In [ ]:
selected

## Estimating the Forward and Finding K0
The forward level is estimated from put-call parity.


In [ ]:
from code.labs.lab07_vix_construction import build_vix_calculation

In [ ]:
calc = build_vix_calculation()

In [ ]:
from code.labs.lab07_vix_construction import term_summary

In [ ]:
term_summary(calc).round(4)

## Building the Out-of-the-Money Strip
The option strip is assembled around latexmath:[K_0].


In [ ]:
near = calc["terms"][0]

In [ ]:
contrib = near.contribution_table

In [ ]:
cols = ["strike", "option_mid", "delta_k", "contribution"]

In [ ]:
contrib[cols].sort_values(
    "contribution", ascending=False
).head(8).round(6)

## Computing the VIX-Style Index Level
After interpolation, the index level is the square root of the thirty-day
annualized variance multiplied by 100:


In [ ]:
calc["variance_30"]

In [ ]:
round(calc["vix"], 2)

## Figure Generation (Optional)
Run the lab figure scripts under `code/figures/` to regenerate the PNG files
under `assets/figures/`.


In [ ]:
# Figure generation code adapted from
# `code/figures/lab07_vix_interpolation.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = PROJECT_ROOT

from code.labs.lab07_vix_construction import TARGET_DAYS
from code.labs.lab07_vix_construction import build_vix_calculation

def main() -> None:
    """Generate the total-variance interpolation figure."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    calc = build_vix_calculation()
    terms = calc["terms"]
    days = [term.days for term in terms]
    total_var = [term.ttm * term.variance for term in terms]
    target_total = calc["variance_30"] * TARGET_DAYS / 365

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(days, total_var, marker="o", label="Listed expiries")
    ax.scatter(
        [TARGET_DAYS],
        [target_total],
        color="tab:red",
        zorder=3,
        label="30-day target",
    )
    ax.set_title("Linear interpolation in total variance")
    ax.set_xlabel("Calendar days to expiry")
    ax.set_ylabel("Total variance")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend()

    fig.tight_layout()

main()

In [ ]:
# Figure generation code adapted from
# `code/figures/lab07_vix_strike_contributions.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = PROJECT_ROOT

from code.labs.lab07_vix_construction import build_vix_calculation

def main() -> None:
    """Generate the strike-contribution figure for the near maturity."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    calc = build_vix_calculation()
    term = calc["terms"][0]
    table = term.contribution_table.copy()
    table["scaled"] = table["contribution"] * 1_000_000

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.bar(table["strike"], table["scaled"], width=12.0, color="tab:blue")
    ax.axvline(term.forward, color="black", linestyle="--", label="Forward")
    ax.axvline(term.k0, color="tab:red", linestyle=":", label="K0")
    ax.set_title("Option-strip contributions to variance")
    ax.set_xlabel("Strike")
    ax.set_ylabel("Contribution × 1,000,000")
    ax.grid(True, axis="y", linestyle="--", alpha=0.3)
    ax.legend()

    fig.tight_layout()

main()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
